# ***Entscheidungsbäume (CART)***
**Methoden der Data Science – Übung, WS 2026/27, Haya Halimeh**

# Einführung
Bisher haben wir mit der linearen Regression eine numerische Zielvariable vorhergesagt. In dieser Übung wechseln wir zur __Klassifikation__: Wir sagen vorher, ob ein Kunde einen Mikrokredit bekommt oder nicht.

Dafür lernen wir drei Dinge:

1. __Entscheidungsbäume__ (Classification Trees) mit `DecisionTreeClassifier` trainieren
2. __Overfitting__ verhindern: mit Hyperparametern und mit Pruning
3. Klassifikationsmodelle __bewerten__: Accuracy, Confusion Matrix und AUC

Die Aufgaben in diesem Notebook gehören zu Übungsblatt 6, Aufgabe 3.

In [ ]:
import numpy as np
import random
import pandas as pd

# Seed für Reproduzierbarkeit setzen
np.random.seed(42)  # Seed für NumPy
random.seed(42) # Seed für das random-Modul

# Daten laden
Zuerst laden wir den MicroCredit-Datensatz. Die Spalte __Decision__ gibt an, ob der Kredit gewährt wurde (1) oder nicht (0).

*Führen Sie den Code unten aus.*

In [ ]:
# Daten aus einer CSV-Datei laden
data = pd.read_csv("https://raw.githubusercontent.com/kbrennig/MODS_WS26_27/refs/heads/main/data/MicroCredit.csv")

data.head()

# Trainings- und Testdaten erstellen

Vor dem Aufteilen entfernen wir die Spalte __ID__. Sie dient nur zur Identifikation und hat keinen Zusammenhang mit der Zielvariable.

Danach teilen wir die Daten wie in den letzten Wochen in 80 % Trainingsdaten und 20 % Testdaten auf.

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.model_selection import train_test_split

# Spalte ID entfernen
data = data.drop(columns="ID")

data_training, data_test = train_test_split(data, test_size=0.2, random_state=42)

# Kategoriale Variablen kodieren

Machine-Learning-Modelle in `sklearn` können nicht mit Text-Kategorien arbeiten, sie brauchen Zahlen. Deshalb kodieren wir die kategorialen Variablen als Dummy-Variablen (One-Hot-Encoding), so wie in der Vorlesung.

Der Code unten macht Folgendes:

1. **Kategoriale Spalten festlegen:** Wir legen eine Liste mit allen kategorialen Spalten an.
2. **Encoder erstellen:**
    * `drop='first'`: Pro Variable wird die erste Kategorie weggelassen, damit keine redundanten Spalten entstehen. Beispiel __Gender__: Die Kategorie *Male* wird weggelassen, es bleibt eine Spalte *Female* (1 = weiblich, 0 = sonst).
    * `handle_unknown='infrequent_if_exist'`: Taucht im Testdatensatz eine Kategorie auf, die es in den Trainingsdaten nicht gab, wird sie einer Sammelkategorie zugeordnet, statt einen Fehler zu erzeugen.
3. **Encoder auf den Trainingsdaten anpassen** (`fit_transform`): Der Encoder lernt die Kategorien nur aus den Trainingsdaten. So gelangen keine Informationen aus den Testdaten ins Modell.
4. **Dummy-Spalten anhängen:** Die neuen Spalten werden an die Trainingsdaten angehängt, die ursprünglichen Spalten entfernt.
5. **Zielvariable festlegen:** `X_train` enthält alle Prädiktoren, `y_train` nur die Zielvariable __Decision__.
6. **Testdaten transformieren** (`transform`): Der auf den Trainingsdaten angepasste Encoder wird auf die Testdaten angewendet. Danach legen wir `X_test` und `y_test` genauso an.

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Kategoriale Spalten
categorical_columns = [
    "Tier",
    "Build_Selfcon",
    "Accommodation_Class",
    "Loan_Type",
    "Gender",
    "Employment_Type",
    "Doc_Proof_Inc",
    "Marital_Status",
    "Employer_Type",
    "Education_Class",
    "Mode_of_origin_class"
]

# OneHotEncoder erstellen
encoder = OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist', sparse_output=False)

# Encoder auf den Trainingsdaten anpassen und anwenden
encoded_training = encoder.fit_transform(data_training.loc[:,categorical_columns])

# DataFrame mit den Dummy-Spalten erstellen
encoded_training_df = pd.DataFrame(encoded_training, columns=encoder.get_feature_names_out(categorical_columns))

# Dummy-Spalten anhängen und ursprüngliche Spalten entfernen
data_training = data_training.reset_index(drop=True).join(encoded_training_df).drop(columns=categorical_columns)

# Prädiktoren und Zielvariable trennen
X_train = data_training.drop(columns="Decision")
y_train = data_training["Decision"]

# Testdaten mit demselben Encoder transformieren
encoded_test = encoder.transform(data_test.loc[:,categorical_columns])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(categorical_columns))
data_test = data_test.reset_index(drop=True).join(encoded_test_df).drop(columns=categorical_columns)
X_test = data_test.drop(columns="Decision")
y_test = data_test["Decision"]

---

# Unbeschränkter Baum (`model_01`)

## Baum trainieren

Mit `DecisionTreeClassifier` erstellen wir einen Klassifikationsbaum und passen ihn mit `.fit()` an die Trainingsdaten an. `X_train` enthält die Prädiktoren, `y_train` die tatsächlichen Werte der Zielvariable.

Wir geben keine Hyperparameter an. Der Baum wächst deshalb so lange, bis die Blätter rein sind oder sich nicht weiter aufteilen lassen.

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_01 = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

## Baum anzeigen

Mit `plot_tree()` können wir einen Baum zeichnen. Jeder Knoten zeigt:

* die Split-Regel (z. B. `Age <= 30.5`; links geht es weiter, wenn die Regel zutrifft),
* `samples`: den Anteil der Trainingsdaten, der in diesem Knoten landet,
* `value`: die Anteile der Klassen [0, 1] in diesem Knoten,
* `class`: die vorhergesagte Klasse. `y[1]` steht für Klasse 1 (Kredit gewährt), `y[0]` für Klasse 0 (abgelehnt).

Der unbeschränkte Baum `model_01` ist so groß, dass man ihn kaum lesen kann. Sie können auf die Grafik klicken, um hinein- und herauszuzoomen.

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# model_01: unbeschränkter Baum
plt.figure(figsize=(120,40))
plot_tree(model_01, feature_names=X_train.columns, impurity=False, filled=True, rounded=True, proportion=True, class_names=True, fontsize=10)
plt.show()

## Vorhersagen auf den Testdaten

Bei der Klassifikation sagt das Modell keine Zahl vorher, sondern für jede Beobachtung eine __Wahrscheinlichkeit__ für jede Klasse. `predict_proba()` gibt diese Wahrscheinlichkeiten zurück, die Spalte `[:,1]` enthält die Wahrscheinlichkeit für Klasse 1 (Kredit gewährt).

Mit einem __Schwellenwert__ machen wir daraus eine Klasse: `np.where(predictions_test_01[:,1] > 0.5, 1, 0)` sagt 1 (gewährt) vorher, wenn die Wahrscheinlichkeit größer als 0,5 ist, sonst 0 (abgelehnt).

*Führen Sie den Code unten aus.*

In [ ]:
# Wahrscheinlichkeiten auf den Testdaten vorhersagen
predictions_test_01 = model_01.predict_proba(X_test)

# Mit Schwellenwert 0,5 in Klassen 0/1 umwandeln
predictions_test_01_binary = np.where(predictions_test_01[:,1] > 0.5, 1, 0)

## Accuracy und Confusion Matrix

Die __Accuracy__ gibt an, welcher Anteil der Vorhersagen richtig ist. Mit `accuracy_score` vergleichen wir die tatsächlichen Werte `y_test` mit den vorhergesagten Klassen `predictions_test_01_binary`.

Genauer sehen wir die Fehler in der __Confusion Matrix__. Bei einer binären Klassifikation gibt es zwei Arten von Fehlern:

1. __False Positives (FP):__ Tatsächlich 0, vorhergesagt 1.
2. __False Negatives (FN):__ Tatsächlich 1, vorhergesagt 0.

In `sklearn` stehen die tatsächlichen Klassen in den Zeilen und die vorhergesagten Klassen in den Spalten:

|  | Vorhergesagt 0 | Vorhergesagt 1 |
|---|---|---|
| **Tatsächlich 0** | TN | FP |
| **Tatsächlich 1** | FN | TP |

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# Accuracy berechnen und ausgeben
accuracy_model_01 = accuracy_score(y_test, predictions_test_01_binary)
print(accuracy_model_01)

# Confusion Matrix anzeigen
ConfusionMatrixDisplay.from_predictions(y_test, predictions_test_01_binary)

---

# Beschränkter Baum (`model_02`)

Ein Nachteil von Entscheidungsbäumen ist, dass sie leicht __overfitten__, wenn man ihre Größe nicht begrenzt: Das Modell passt sich zu stark an die Trainingsdaten an und funktioniert auf neuen Daten schlechter. Das sieht man an `model_01`.

Eine Lösung ist, das Wachstum des Baums mit __Hyperparametern__ zu begrenzen. Hyperparameter legen wir selbst fest, das Modell lernt sie nicht aus den Daten:

* `max_depth`: Maximale Tiefe des Baums. Der Wurzelknoten hat die Tiefe 0.
* `min_samples_split`: Mindestanzahl an Beobachtungen, die ein Knoten haben muss, damit er weiter aufgeteilt werden darf.
* `min_samples_leaf`: Mindestanzahl an Beobachtungen in jedem Blatt (Endknoten).

Die Hyperparameter übergeben wir als Argumente an `DecisionTreeClassifier()`, zum Beispiel `DecisionTreeClassifier(max_depth=3, random_state=42)`.

## **Ihre Aufgabe 1** (Übungsblatt 6, Aufgabe 3, Teil a)

Trainieren Sie auf den Trainingsdaten einen beschränkten Baum und speichern Sie ihn als `model_02`. Der Baum soll diese Vorgaben erfüllen:

* Ein Knoten muss mindestens __60__ Beobachtungen haben, damit er weiter aufgeteilt werden darf.
* In jedem Blatt müssen mindestens __20__ Beobachtungen liegen.
* Der Baum darf höchstens __5__ Ebenen tief sein.

Verwenden Sie wie bei `model_01` außerdem `random_state=42`.

*Tipp: Die passenden Namen der Hyperparameter finden Sie in der Liste oben. Die Zelle danach zeigt Ihren Baum an. Vergleichen Sie ihn mit `model_01` und beantworten Sie damit Übungsblatt 6, Aufgabe 3, Teil a.*

In [ ]:
# Ihr Code hier!

In [ ]:
# model_02 anzeigen (funktioniert, sobald Sie model_02 oben erstellt haben)
plt.figure(figsize=(20,10))
plot_tree(model_02, feature_names=X_train.columns, impurity=False, filled=True, rounded=True, proportion=True, class_names=True, fontsize=10)
plt.show()

## **Ihre Aufgabe 2** (Übungsblatt 6, Aufgabe 3, Teil b)

Bewerten Sie jetzt `model_02` genauso wie `model_01`:

1. Sagen Sie die Wahrscheinlichkeiten auf den Testdaten vorher und speichern Sie sie als `predictions_test_02`.
2. Wandeln Sie sie mit dem Schwellenwert 0,5 in Klassen um und speichern Sie das Ergebnis als `predictions_test_02_binary`.
3. Berechnen Sie die Accuracy und zeigen Sie die Confusion Matrix an.

*Tipp: Kopieren Sie den Code von `model_01` oben und ersetzen Sie `01` durch `02`.*

Beantworten Sie mit dem Ergebnis Übungsblatt 6, Aufgabe 3, Teil b i) und ii).

In [ ]:
# Ihr Code hier!

---

# ROC-Kurve und AUC

Accuracy und Confusion Matrix bewerten das Modell nur für __einen__ Schwellenwert (hier 0,5). Die __ROC-Kurve__ betrachtet dagegen __alle__ Schwellenwerte: Für jeden Schwellenwert wird die True Positive Rate (TPR) gegen die False Positive Rate (FPR) abgetragen. Die __AUC__ (Area Under the Curve) ist die Fläche unter dieser Kurve.

Deshalb brauchen wir hier die vorhergesagten __Wahrscheinlichkeiten__ für Klasse 1, nicht die 0/1-Vorhersagen.

* `roc_auc_score()` berechnet die AUC. Sie liegt zwischen 0 und 1: Je näher an 1, desto besser trennt das Modell die beiden Klassen. 0,5 entspricht Raten.
* `RocCurveDisplay.from_predictions()` plottet die ROC-Kurve. Mit `plot_chance_level=True` wird zusätzlich die Diagonale von (0, 0) nach (1, 1) eingezeichnet. Sie steht für zufälliges Raten:
    * Kurve über der Diagonale → Modell ist besser als Raten
    * Kurve nahe der Diagonale → Modell ist schwach
    * Kurve unter der Diagonale → Modell sagt systematisch das Gegenteil vorher

*Führen Sie den Code unten aus.*

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

# AUC von model_01 berechnen und ausgeben
auc_score = roc_auc_score(y_test, predictions_test_01[:,1])
print("AUC Score:", auc_score)

# ROC-Kurve plotten
RocCurveDisplay.from_predictions(y_test, predictions_test_01[:,1], plot_chance_level=True)

## AUC für `model_02`

Zum Vergleich berechnen wir die AUC und die ROC-Kurve auch für `model_02`. Die Zelle funktioniert, sobald Sie oben `predictions_test_02` erstellt haben.

*Führen Sie den Code unten aus und beantworten Sie damit Übungsblatt 6, Aufgabe 3, Teil b iii).*

In [ ]:
# AUC von model_02 berechnen (mit den Wahrscheinlichkeiten für Klasse 1)
auc_score_02 = roc_auc_score(y_test, predictions_test_02[:,1])
print("AUC Score model_02:", auc_score_02)

# ROC-Kurve plotten
RocCurveDisplay.from_predictions(y_test, predictions_test_02[:,1], plot_chance_level=True)

---

# Zusatz (freiwillig): Pruning (`model_03`)

Eine zweite Möglichkeit ist das __Pruning__: Der Baum wächst zuerst groß und wird dann zurückgeschnitten. Dafür setzen wir den Parameter `ccp_alpha`. Vereinfacht gesagt gibt er an, wie stark ein Split das Modell mindestens verbessern muss, damit er im Baum bleibt.

Vergleichen Sie den Baum mit `model_01` und `model_02`.

*Führen Sie den Code unten aus.*

In [ ]:
# Baum mit Pruning trainieren
model_03 = DecisionTreeClassifier(ccp_alpha=0.01, random_state=42).fit(X_train, y_train)

# Baum anzeigen
plt.figure(figsize=(10,5))
plot_tree(model_03, feature_names=X_train.columns, impurity=False, filled=True, rounded=True, proportion=True, class_names=True, fontsize=10)
plt.show()

---

# Zusammenfassung

Was haben wir in dieser Übung gemacht?

1. Wir haben mit __Entscheidungsbäumen__ eine kategoriale Zielvariable vorhergesagt.
2. Wir haben zwei Wege kennengelernt, __Overfitting__ zu verringern: Hyperparameter und Pruning.
3. Wir haben Klassifikationsmodelle mit __Accuracy__, __Confusion Matrix__ und __AUC__ bewertet.

*Freiwillig: In der Zelle unten können Sie ausprobieren, wie sich andere Hyperparameter auf den Baum und die AUC auswirken.*

In [ ]:
# Ihr Code hier!